# NILM-Engine — Run 7: SPEED_GROUP 분리 (fast / slow / always_on)

| 항목 | 값 |
|------|----||
| 실험 전략 | 3개 그룹별 독립 모델 + 그룹별 EXP1→2→3→4 incremental |
| 그룹 구성 | fast(12종, 30Hz@1024) · slow(5종, 1Hz@1800) · always_on(5종, 1Hz@1800) |
| Early stopping | val_mae 단독(↓) — Run6와 동일 |
| Loss masking | 그룹 외 가전 appliance_scale=0 (MSE) + pos_weight=0 (BCE) |
| appliance_loss_scale | fast: 인덕션×2.0 · slow: 의류건조기×2.0, 세탁기×2.0 |
| Best 선택 | 그룹별 val MAE 기준 최적 주차 체크포인트 사용 |
| Test 평가 | 3 모델 각자 평가 → 가전별 지표 merge |
| Train | house_011,015,016,017,033,039,054,063 (주차 누적) |
| Val | house_049 (전체 기간) |
| Test | house_067 (전체 기간) |
| 비교 기준 | Run6 val=34.67W / test=40.76W |

In [ ]:
!pip install -q gudhi

In [ ]:
from google.colab import auth
auth.authenticate_user()
print('GCP 인증 완료')

In [ ]:
import gcsfs
gcs = gcsfs.GCSFileSystem()
print('GCS 연결 완료')

In [ ]:
SPLIT = {
    'train': ['house_011', 'house_015', 'house_016', 'house_017',
              'house_033', 'house_039', 'house_054', 'house_063'],
    'val':   ['house_049'],
    'test':  ['house_067'],
}
BUCKET_PREFIX = 'ax-nilm-data-dhwang0803-us/nilm/training_dev10'
LABEL_PATH    = 'ax-nilm-data-dhwang0803-us/nilm/labels/training.parquet'

# 그룹별 appliance_loss_scale (그룹 내 가전만 지정, 나머지는 1.0)
APPLIANCE_LOSS_SCALE = {
    'fast':      {'인덕션(전기레인지)': 2.0},
    'slow':      {'의류건조기': 2.0, '세탁기': 2.0},
    'always_on': {},
}

print(f"Train {len(SPLIT['train'])}개 / Val {len(SPLIT['val'])}개 / Test {len(SPLIT['test'])}개")

In [ ]:
import os, sys

REPO_DIR = '/content/ax_nilm'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/dhwang0803-glitch/ax_nilm {REPO_DIR} -q
    print('클론 완료')
else:
    !git -C {REPO_DIR} pull -q && echo '최신화 완료'

SRC_DIR     = f'{REPO_DIR}/nilm-engine/src'
SCRIPTS_DIR = f'{REPO_DIR}/nilm-engine/scripts'
CFG_DIR     = f'{REPO_DIR}/nilm-engine/config'

for d in [SRC_DIR, SCRIPTS_DIR]:
    if d not in sys.path:
        sys.path.insert(0, d)
print('경로 설정 완료')

In [ ]:
import yaml

from classifier.label_map import (
    APPLIANCE_LABELS, SPEED_GROUP, SPEED_GROUP_CONFIG,
)
from acquisition.gcs_loader import (
    GCSNILMDataset, build_house_mask_dict,
)
from acquisition.preprocessor import PowerScaler
from train_model import (
    build_model, evaluate, train_one_epoch,
    compute_pos_weight, _NILMDatasetWithTDA,
    _merge_group_metrics,
)

with open(f'{CFG_DIR}/train.yaml')   as f: TRAIN_CFG   = yaml.safe_load(f)
with open(f'{CFG_DIR}/dataset.yaml') as f: DATASET_CFG = yaml.safe_load(f)

# 그룹별 가전 인덱스 (APPLIANCE_LABELS 기준)
_label_idx = {name: i for i, name in enumerate(APPLIANCE_LABELS)}
GROUP_INDICES = {'fast': [], 'slow': [], 'always_on': []}
for name, group in SPEED_GROUP.items():
    GROUP_INDICES[group].append(_label_idx[name])

# 그룹별 가전 이름 집합 (per_appliance 필터링용)
GROUP_NAMES = {
    g: {APPLIANCE_LABELS[i] for i in idxs}
    for g, idxs in GROUP_INDICES.items()
}

print('그룹별 가전 인덱스:')
for g, idxs in GROUP_INDICES.items():
    names = [APPLIANCE_LABELS[i] for i in sorted(idxs)]
    cfg   = SPEED_GROUP_CONFIG[g]
    print(f'  {g:10s} ({len(idxs)}종, {cfg["resample_hz"]}Hz@{cfg["window_size"]}): {names}')

print('\nEXP 설정:')
for exp in ['EXP1', 'EXP2', 'EXP3', 'EXP4']:
    cfg = TRAIN_CFG['experiments'][exp]
    print(f'  {exp}: week={cfg["week"]}  resume_from={cfg["resume_from"]}')

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

CKPT_DIR       = Path('/content/drive/MyDrive/nilm_run7/checkpoints')
RESULTS_DIR    = Path('/content/drive/MyDrive/nilm_run7/results')
CACHE_DIR_FAST = Path('/content/drive/MyDrive/nilm/cache')        # Run6 캐시 재사용 (30Hz@1024)
CACHE_DIR_SLOW = Path('/content/drive/MyDrive/nilm_run7/cache_1hz')  # 신규 (1Hz@1800)

for d in [CKPT_DIR, RESULTS_DIR, CACHE_DIR_FAST, CACHE_DIR_SLOW]:
    d.mkdir(parents=True, exist_ok=True)

# 그룹 → 캐시 디렉터리 매핑
GROUP_CACHE = {
    'fast':      CACHE_DIR_FAST,
    'slow':      CACHE_DIR_SLOW,
    'always_on': CACHE_DIR_SLOW,
}

print(f'체크포인트  → {CKPT_DIR}')
print(f'결과        → {RESULTS_DIR}')
print(f'캐시(fast)  → {CACHE_DIR_FAST}')
print(f'캐시(slow)  → {CACHE_DIR_SLOW}')

## 가구별 등록 가전 마스크 빌드

In [ ]:
HOUSE_MASK = build_house_mask_dict(gcs, LABEL_PATH)

print(f'마스크 빌드 완료: {len(HOUSE_MASK)}가구')
all_houses = SPLIT['train'] + SPLIT['val'] + SPLIT['test']
for h in all_houses:
    mask = HOUSE_MASK.get(h)
    if mask is None:
        print(f'  {h}: ⚠️ 마스크 없음')
        continue
    registered = [APPLIANCE_LABELS[i] for i, v in enumerate(mask) if v > 0]
    print(f'  {h} ({int(mask.sum())}종): {registered}')

## 캐시 빌드 (CPU 런타임에서 실행)

- **fast 그룹**: Run6 캐시(`nilm/cache`) 재사용 — 이미 빌드됨
- **slow / always_on**: 1Hz 다운샘플 + window=1800 — 신규 빌드 필요

In [ ]:
# fast 캐시 확인 (Run6와 동일 파라미터 → 재사용)
import hashlib
from features.tda import TDA_DIM

_fast_cfg = SPEED_GROUP_CONFIG['fast']
ws_f = _fast_cfg['window_size']
st_f = _fast_cfg['stride']
ec   = DATASET_CFG['window'].get('event_context', 0)
ss   = DATASET_CFG['window'].get('steady_stride', 0)

def _week_key(week):
    return hashlib.md5(f'None|{week}|None|{BUCKET_PREFIX}|True'.encode()).hexdigest()[:8]

def _tda_key(houses, week, ws, st, hz):
    return hashlib.md5(
        f'{sorted(houses)}|None|{week}|None|{ws}|{st}|{BUCKET_PREFIX}|{hz}|True'.encode()
    ).hexdigest()[:12]

print('=== fast 캐시 확인 (Run6 재사용) ===')
all_ok = True
for label, houses, week in [
    ('train_w1', SPLIT['train'], 1), ('train_w2', SPLIT['train'], 2),
    ('train_w3', SPLIT['train'], 3), ('train_w4', SPLIT['train'], 4),
    ('val',      SPLIT['val'],   None), ('test',    SPLIT['test'],  None),
]:
    wk   = _week_key(week)
    b_ok = all((CACHE_DIR_FAST / f'nilm_gcs_{h}_{wk}.npz').exists() for h in houses)
    t_key = _tda_key(houses, week, ws_f, st_f, 30)
    t_ok  = (CACHE_DIR_FAST / f'tda_{t_key}_ec{ec}_ss{ss}_d{TDA_DIM}.pt').exists()
    flag  = '✅' if (b_ok and t_ok) else '❌'
    print(f'  {flag}  {label:10s}  base={"O" if b_ok else "X"}  tda={"O" if t_ok else "X"}')
    if not (b_ok and t_ok):
        all_ok = False

if not all_ok:
    print('\n⚠️ fast 캐시 누락 → CPU에서 Run6 캐시 빌드 셀 먼저 실행')
else:
    print('\n✅ fast 캐시 모두 존재 — 재사용 가능')

In [ ]:
import gc

_slow_cfg = SPEED_GROUP_CONFIG['slow']   # resample_hz=1, window_size=1800, stride=30
ws_s = _slow_cfg['window_size']
st_s = _slow_cfg['stride']
hz_s = _slow_cfg['resample_hz']

_ds_slow_common = dict(
    gcs_fs=gcs,
    bucket_prefix=BUCKET_PREFIX, label_path=LABEL_PATH,
    window_size=ws_s, stride=st_s,
    event_context=ec, steady_stride=ss,
    cache_dir=CACHE_DIR_SLOW, denoise=True,
    house_mask_dict=HOUSE_MASK,
    resample_hz=hz_s,
)

print(f'slow/always_on 캐시 빌드: {hz_s}Hz @ window={ws_s}')
for split_name, houses, weeks in [
    ('train', SPLIT['train'], [1, 2, 3, 4]),
    ('val',   SPLIT['val'],   [None]),
    ('test',  SPLIT['test'],  [None]),
]:
    for w in weeks:
        label = f'{split_name}(week={w})'
        print(f'\n[{label}] 빌드 중...')
        base = GCSNILMDataset(
            houses, week=w,
            fit_scaler=(split_name == 'train' and w == 1),
            **_ds_slow_common,
        )
        print(f'  base: {len(base):,} windows')
        tda = _NILMDatasetWithTDA(base, cache_dir=CACHE_DIR_SLOW)
        print(f'  TDA:  완료')
        del tda, base
        gc.collect()

print('\n✅ slow/always_on 캐시 빌드 완료')

## GPU 투입 전 캐시 확인

> GPU 런타임으로 전환 후 이 셀을 먼저 실행한다.

In [ ]:
import hashlib
from features.tda import TDA_DIM

_fast_cfg = SPEED_GROUP_CONFIG['fast']
_slow_cfg = SPEED_GROUP_CONFIG['slow']
ec = DATASET_CFG['window'].get('event_context', 0)
ss = DATASET_CFG['window'].get('steady_stride', 0)

def _check_cache(houses, week, ws, st, hz, cache_dir):
    wk   = hashlib.md5(f'None|{week}|None|{BUCKET_PREFIX}|True'.encode()).hexdigest()[:8]
    b_ok = all((cache_dir / f'nilm_gcs_{h}_{wk}.npz').exists() for h in houses)
    t_key = hashlib.md5(
        f'{sorted(houses)}|None|{week}|None|{ws}|{st}|{BUCKET_PREFIX}|{hz}|True'.encode()
    ).hexdigest()[:12]
    t_ok  = (cache_dir / f'tda_{t_key}_ec{ec}_ss{ss}_d{TDA_DIM}.pt').exists()
    return b_ok, t_ok

missing = []
print('=== fast 캐시 (30Hz@1024) ===')
for label, houses, week in [
    ('train_w1', SPLIT['train'], 1), ('train_w2', SPLIT['train'], 2),
    ('train_w3', SPLIT['train'], 3), ('train_w4', SPLIT['train'], 4),
    ('val',      SPLIT['val'],   None), ('test',    SPLIT['test'],  None),
]:
    b_ok, t_ok = _check_cache(houses, week,
                               _fast_cfg['window_size'], _fast_cfg['stride'], 30,
                               CACHE_DIR_FAST)
    flag = '✅' if (b_ok and t_ok) else '❌'
    print(f'  {flag}  {label:10s}  base={"O" if b_ok else "X"}  tda={"O" if t_ok else "X"}')
    if not (b_ok and t_ok): missing.append(f'fast/{label}')

print('\n=== slow/always_on 캐시 (1Hz@1800) ===')
for label, houses, week in [
    ('train_w1', SPLIT['train'], 1), ('train_w2', SPLIT['train'], 2),
    ('train_w3', SPLIT['train'], 3), ('train_w4', SPLIT['train'], 4),
    ('val',      SPLIT['val'],   None), ('test',    SPLIT['test'],  None),
]:
    b_ok, t_ok = _check_cache(houses, week,
                               _slow_cfg['window_size'], _slow_cfg['stride'], 1,
                               CACHE_DIR_SLOW)
    flag = '✅' if (b_ok and t_ok) else '❌'
    print(f'  {flag}  {label:10s}  base={"O" if b_ok else "X"}  tda={"O" if t_ok else "X"}')
    if not (b_ok and t_ok): missing.append(f'slow/{label}')

if missing:
    raise RuntimeError(
        f'\n⛔ 캐시 누락: {missing}\n'
        'GPU 런타임을 끄고 CPU 런타임에서 캐시 빌드 셀을 재실행하세요.'
    )
else:
    print('\n✅ 모든 캐시 확인 완료 — GPU 학습 진행 가능')

## 함수 정의

**핵심 변경점 (Run7)**:
- `group_name` / `group_indices` / `group_cfg` 추가 — 그룹별 window_size / resample_hz 분기
- `appliance_scale[non_group] = 0` — 그룹 외 가전 MSE loss 제외
- `pos_weight[non_group] = 0` — 그룹 외 가전 BCE loss 제외
- `always_on` 그룹은 pos_weight=None (분류 BCE 없이 회귀만)
- 체크포인트 파일명: `{EXP}_{model}_{group}.pt`

In [ ]:
import json, time
import torch
import numpy as np
from torch.utils.data import DataLoader
from classifier.label_map import get_on_thresholds as _get_on_thr


def run_exp_gcs_group(
    group_name: str,          # 'fast' | 'slow' | 'always_on'
    exp_name: str,            # 'EXP1' ~ 'EXP4'
    model_name: str = 'cnn_tda',
    denoise: bool = True,
    val_week=None,
) -> dict:
    """그룹별 incremental 학습. early stopping = val_mae 단독(↓)."""

    group_cfg    = SPEED_GROUP_CONFIG[group_name]
    group_idx    = sorted(GROUP_INDICES[group_name])
    cache_dir    = GROUP_CACHE[group_name]

    ws  = group_cfg['window_size']
    st  = group_cfg['stride']
    hz  = group_cfg['resample_hz']
    ec  = DATASET_CFG['window'].get('event_context')
    ss_ = DATASET_CFG['window'].get('steady_stride')

    exp_cfg    = TRAIN_CFG['experiments'][exp_name]
    week       = exp_cfg.get('week')
    bs         = TRAIN_CFG['training']['batch_size']
    ep         = TRAIN_CFG['training']['epochs']
    pat        = TRAIN_CFG['training']['early_stopping_patience']
    lr         = TRAIN_CFG['training']['learning_rate']
    wd         = TRAIN_CFG['optimizer']['weight_decay']
    lambda_mse = TRAIN_CFG['training']['loss_weights']['mse']
    pos_w_max  = float(TRAIN_CFG['training'].get('pos_weight_max', 20.0))

    print(f'\n{"="*60}')
    print(f'  [{group_name}] {exp_name}  week={week}  {hz}Hz@{ws}')
    print(f'{"="*60}')

    # ── scaler 로드 (resume) ──────────────────────────────────────────────────
    resume_exp   = exp_cfg.get('resume_from')
    prev_scaler  = None
    if resume_exp:
        sc_path = CKPT_DIR / f'{resume_exp}_{model_name}_{group_name}_scaler.json'
        if sc_path.exists():
            prev_scaler = PowerScaler.load(sc_path)
            print(f'  scaler 로드: mean={prev_scaler.mean:.2f}W  std={prev_scaler.std:.2f}W')

    # ── 데이터셋 ──────────────────────────────────────────────────────────────
    _ds_common = dict(
        gcs_fs=gcs,
        bucket_prefix=BUCKET_PREFIX, label_path=LABEL_PATH,
        window_size=ws, stride=st,
        event_context=ec, steady_stride=ss_,
        cache_dir=cache_dir, denoise=denoise,
        house_mask_dict=HOUSE_MASK,
        resample_hz=hz,
    )
    if prev_scaler is not None:
        base_train = GCSNILMDataset(SPLIT['train'], scaler=prev_scaler, week=week, **_ds_common)
    else:
        base_train = GCSNILMDataset(SPLIT['train'], fit_scaler=True, week=week, **_ds_common)

    base_val  = GCSNILMDataset(SPLIT['val'], scaler=base_train.scaler, week=val_week, **_ds_common)
    train_ds  = _NILMDatasetWithTDA(base_train, cache_dir=cache_dir)
    val_ds    = _NILMDatasetWithTDA(base_val,   cache_dir=cache_dir)
    train_dl  = DataLoader(train_ds, batch_size=bs, shuffle=True,  num_workers=2, pin_memory=False)
    val_dl    = DataLoader(val_ds,   batch_size=bs, shuffle=False, num_workers=2, pin_memory=False)
    print(f'  train(week{week}): {len(train_ds):,} windows  |  val: {len(val_ds):,} windows')

    _raw_thr   = np.array(_get_on_thr(), dtype=np.float32)
    _scl       = base_train.scaler
    _fixed_thr = (_raw_thr - _scl.mean) / _scl.std if _scl is not None else None

    # ── 모델 ─────────────────────────────────────────────────────────────────
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'  device: {device}')
    model = build_model(model_name, ws).to(device)

    if resume_exp:
        prev_ckpt = CKPT_DIR / f'{resume_exp}_{model_name}_{group_name}.pt'
        if prev_ckpt.exists():
            _ckpt  = torch.load(prev_ckpt, map_location=device, weights_only=True)
            _state = _ckpt['model_state'] if isinstance(_ckpt, dict) and 'model_state' in _ckpt else _ckpt
            model.load_state_dict(_state)
            print(f'  모델 로드: {prev_ckpt.name}')
        else:
            print(f'  경고: {prev_ckpt.name} 없음 — 처음부터 학습')

    optimizer  = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, 'min',
        factor=TRAIN_CFG['scheduler']['factor'],
        patience=TRAIN_CFG['scheduler']['patience'],
    )
    amp_scaler = torch.amp.GradScaler('cuda') if device.type == 'cuda' else None

    # ── appliance_scale: 그룹 외 가전 MSE loss 제외 ───────────────────────────
    _app_idx_map = {name: i for i, name in enumerate(APPLIANCE_LABELS)}
    appliance_scale = torch.zeros(len(APPLIANCE_LABELS), device=device)  # 기본 0 (비그룹 제외)
    for i in group_idx:
        appliance_scale[i] = 1.0
    # 그룹 내 loss scale 추가
    for name, s in APPLIANCE_LOSS_SCALE.get(group_name, {}).items():
        if name in _app_idx_map:
            appliance_scale[_app_idx_map[name]] = float(s)
            print(f'  appliance_loss_scale [{name}]: ×{s}')

    # ── pos_weight: 그룹 외 가전 BCE loss 제외 ────────────────────────────────
    print('  pos_weight 계산 중...')
    _pw_raw = compute_pos_weight(train_dl, device, max_weight=pos_w_max)
    if group_name == 'always_on':
        pos_weight = None   # always_on은 분류 BCE 없이 회귀만
    else:
        pos_weight = _pw_raw.clone()
        non_group  = [i for i in range(len(APPLIANCE_LABELS)) if i not in group_idx]
        pos_weight[non_group] = 0.0  # 비그룹 BCE 기여 제거

    # ── 학습 루프 (early stopping: val_mae 단독) ───────────────────────────────
    best_score          = float('inf')
    best_cls_thresholds = None
    best_vm             = None
    best_state          = None
    no_improve          = 0
    t0 = time.perf_counter()

    for epoch in range(1, ep + 1):
        loss = train_one_epoch(model, train_dl, optimizer, model_name, device,
                               amp_scaler=amp_scaler, pos_weight=pos_weight,
                               lambda_mse=lambda_mse, appliance_scale=appliance_scale)
        vm   = evaluate(model, val_dl, model_name, device, cls_thresholds=_fixed_thr)
        scheduler.step(vm['mae'])
        lr_now = optimizer.param_groups[0]['lr']
        f1s = f"  f1_cls={vm['f1_cls']:.3f}" if vm.get('f1_cls') is not None else ''
        print(f'  ep {epoch:3d}/{ep}  loss={loss:.4f}  '
              f'val_mae={vm["mae"]:.2f}W  f1={vm["f1"]:.3f}{f1s}  lr={lr_now:.2e}')

        _score = vm['mae']
        if _score < best_score or best_state is None:
            best_score          = _score
            best_cls_thresholds = vm.get('best_cls_thresholds')
            best_vm             = vm
            best_state          = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve          = 0
        else:
            no_improve += 1
            if no_improve >= pat:
                print(f'  조기 종료 ({pat} epoch 개선 없음)')
                break

    elapsed = time.perf_counter() - t0

    # ── 저장 ──────────────────────────────────────────────────────────────────
    if best_state:
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})

    torch.save({'model_state': model.state_dict(), 'best_cls_thresholds': best_cls_thresholds},
               CKPT_DIR / f'{exp_name}_{model_name}_{group_name}.pt')
    if base_train.scaler is not None:
        base_train.scaler.save(CKPT_DIR / f'{exp_name}_{model_name}_{group_name}_scaler.json')

    fm = best_vm if best_vm is not None else vm
    fm.update({'exp': exp_name, 'model': model_name, 'group': group_name,
               'week': week, 'denoise': denoise,
               'training_time_s': round(elapsed, 1),
               'final_lr': optimizer.param_groups[0]['lr']})
    with open(RESULTS_DIR / f'{exp_name}_{model_name}_{group_name}_metrics.json', 'w') as f:
        json.dump(fm, f, ensure_ascii=False, indent=2)

    f1s = f"  F1_cls={fm['f1_cls']:.3f}" if fm.get('f1_cls') is not None else ''
    print(f'  ✅ [{group_name}] val MAE={fm["mae"]:.2f}W  F1={fm["f1"]:.3f}{f1s}  ({elapsed:.0f}s)')
    return fm


print('함수 정의 완료 — run_exp_gcs_group')

## Fast 그룹 학습 (12종, 30Hz@1024)

> TV, 전기포트, 선풍기, 헤어드라이기, 에어프라이어, 진공청소기, 전자레인지, 인덕션, 전기장판, 온수매트, 컴퓨터, 전기다리미

In [ ]:
fast_exp1 = run_exp_gcs_group('fast', 'EXP1')

In [ ]:
fast_exp2 = run_exp_gcs_group('fast', 'EXP2')

In [ ]:
fast_exp3 = run_exp_gcs_group('fast', 'EXP3')

In [ ]:
fast_exp4 = run_exp_gcs_group('fast', 'EXP4')

## Slow 그룹 학습 (5종, 1Hz@1800)

> 의류건조기, 전기밥솥, 식기세척기/건조기, 세탁기, 에어컨

> 1Hz 다운샘플 → window=1800 (30분) — 코스 사이클 전체 캡처

In [ ]:
slow_exp1 = run_exp_gcs_group('slow', 'EXP1')

In [ ]:
slow_exp2 = run_exp_gcs_group('slow', 'EXP2')

In [ ]:
slow_exp3 = run_exp_gcs_group('slow', 'EXP3')

In [ ]:
slow_exp4 = run_exp_gcs_group('slow', 'EXP4')

## Always-on 그룹 학습 (5종, 1Hz@1800)

> 제습기, 공기청정기, 일반 냉장고, 김치냉장고, 무선공유기/셋톱박스

> 분류 BCE 없이 회귀만 학습 (pos_weight=None)

In [ ]:
always_exp1 = run_exp_gcs_group('always_on', 'EXP1')

In [ ]:
always_exp2 = run_exp_gcs_group('always_on', 'EXP2')

In [ ]:
always_exp3 = run_exp_gcs_group('always_on', 'EXP3')

In [ ]:
always_exp4 = run_exp_gcs_group('always_on', 'EXP4')

## 최적 주차 선택 (그룹별)

In [ ]:
import json as _json

all_results = {
    'fast':      {'EXP1': fast_exp1,   'EXP2': fast_exp2,   'EXP3': fast_exp3,   'EXP4': fast_exp4},
    'slow':      {'EXP1': slow_exp1,   'EXP2': slow_exp2,   'EXP3': slow_exp3,   'EXP4': slow_exp4},
    'always_on': {'EXP1': always_exp1, 'EXP2': always_exp2, 'EXP3': always_exp3, 'EXP4': always_exp4},
}

best_exps = {}   # group → best EXP name
for group, week_res in all_results.items():
    best_exp = min(week_res, key=lambda k: week_res[k]['mae'])
    best_mae = week_res[best_exp]['mae']
    best_exps[group] = best_exp

    print(f'\n[{group}] 주차별 val MAE:')
    prev = None
    for exp, r in week_res.items():
        trend  = ''
        if prev is not None:
            d = r['mae'] - prev
            trend = f'  (↓{abs(d):.2f}W)' if d < 0 else f'  (↑{abs(d):.2f}W)'
        marker = ' ← best' if exp == best_exp else ''
        print(f'  {exp}: {r["mae"]:.2f}W{trend}{marker}')
        prev = r['mae']

summary = {
    g: {
        'best_exp': best_exps[g],
        'best_val_mae': all_results[g][best_exps[g]]['mae'],
        'per_week': {exp: {'val_mae': r['mae'], 'val_f1': r['f1']}
                     for exp, r in all_results[g].items()},
    }
    for g in all_results
}
with open(RESULTS_DIR / 'run7_week_summary.json', 'w') as _f:
    _json.dump(summary, _f, ensure_ascii=False, indent=2)
print('\n저장: run7_week_summary.json')
print('최적 체크포인트:', {g: best_exps[g] for g in best_exps})

## Test 평가 (그룹별 → merge)

> 각 그룹 최적 체크포인트로 test 평가 → 가전별 지표를 그룹 기준으로 합산

In [ ]:
import json as _json
import torch
import numpy as np
from torch.utils.data import DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
group_test_metrics = {}

for group_name in ['fast', 'slow', 'always_on']:
    best_exp  = best_exps[group_name]
    group_cfg = SPEED_GROUP_CONFIG[group_name]
    cache_dir = GROUP_CACHE[group_name]
    ws  = group_cfg['window_size']
    st  = group_cfg['stride']
    hz  = group_cfg['resample_hz']
    ec  = DATASET_CFG['window'].get('event_context')
    ss_ = DATASET_CFG['window'].get('steady_stride')
    bs  = TRAIN_CFG['training']['batch_size']

    ckpt_path   = CKPT_DIR / f'{best_exp}_cnn_tda_{group_name}.pt'
    scaler_path = CKPT_DIR / f'{best_exp}_cnn_tda_{group_name}_scaler.json'

    if not ckpt_path.exists():
        print(f'[{group_name}] 체크포인트 없음: {ckpt_path}')
        continue

    scaler = PowerScaler.load(scaler_path) if scaler_path.exists() else None
    base_test = GCSNILMDataset(
        SPLIT['test'], scaler=scaler, week=None,
        gcs_fs=gcs, bucket_prefix=BUCKET_PREFIX, label_path=LABEL_PATH,
        window_size=ws, stride=st, event_context=ec, steady_stride=ss_,
        cache_dir=cache_dir, denoise=True,
        house_mask_dict=HOUSE_MASK, resample_hz=hz,
    )
    test_ds = _NILMDatasetWithTDA(base_test, cache_dir=cache_dir)
    test_dl = DataLoader(test_ds, batch_size=bs, shuffle=False, num_workers=2, pin_memory=False)

    model = build_model('cnn_tda', ws).to(device)
    _ckpt  = torch.load(ckpt_path, map_location=device, weights_only=True)
    _state = _ckpt['model_state'] if isinstance(_ckpt, dict) and 'model_state' in _ckpt else _ckpt
    _best_thr = _ckpt.get('best_cls_thresholds') if isinstance(_ckpt, dict) else None
    model.load_state_dict(_state)

    _oracle_thr = np.array(_best_thr, dtype=np.float32) if _best_thr is not None else None

    print(f'\n{"="*60}')
    print(f'  [{group_name}] Test — {best_exp} / oracle threshold')
    print(f'{"="*60}')

    tm = evaluate(model, test_dl, 'cnn_tda', device, cls_thresholds=_oracle_thr)
    tm.update({'exp': best_exp, 'model': 'cnn_tda', 'group': group_name, 'split': 'test'})

    out_path = RESULTS_DIR / f'{best_exp}_cnn_tda_{group_name}_test_oracle_metrics.json'
    with open(out_path, 'w') as _f:
        _json.dump(tm, _f, ensure_ascii=False, indent=2)

    group_test_metrics[group_name] = tm
    f1s = f"  F1_cls={tm['f1_cls']:.3f}" if tm.get('f1_cls') is not None else ''
    print(f'  MAE={tm["mae"]:.2f}W  RMSE={tm["rmse"]:.2f}W  F1={tm["f1"]:.3f}{f1s}')
    print(f'  저장: {out_path.name}')

## 결과 요약 (Run6 비교)

In [ ]:
import json as _json

RUN6_VAL_MAE  = 34.67
RUN6_TEST_MAE = 40.76

print('=' * 60)
print('  Run 7 (SPEED_GROUP 분리) 결과')
print('=' * 60)

# ── 그룹별 val MAE ────────────────────────────────────────────
print('\n그룹별 val MAE (best EXP):')
for g, info in summary.items():
    print(f'  {g:10s}: {info["best_val_mae"]:.2f}W  ({info["best_exp"]})')

# ── 그룹 test metrics를 그룹 소속 가전만 남기도록 필터링 ──────
# evaluate()는 22개 전체 per_appliance를 반환하지만,
# 그룹 모델이 학습한 가전만 신뢰할 수 있으므로 나머지 제거
filtered_group_metrics = {}
for g, tm in group_test_metrics.items():
    g_names = GROUP_NAMES[g]
    filtered_pa = {
        name: metrics
        for name, metrics in tm.get('per_appliance', {}).items()
        if name in g_names
    }
    filtered_group_metrics[g] = {**tm, 'per_appliance': filtered_pa}

# ── _merge_group_metrics: 3개 그룹 per_appliance → 전체 22종 통합 ──
merged = _merge_group_metrics(filtered_group_metrics)

# ── 최종 통합 지표 출력 ────────────────────────────────────────
print('\n' + '=' * 60)
print('  최종 통합 Test 지표 (3 모델 합산, oracle threshold)')
print('=' * 60)
f1s = f"  F1_cls={merged['f1_cls']:.3f}" if merged.get('f1_cls') is not None else ''
print(f'  MAE={merged["mae"]:.2f}W  RMSE={merged["rmse"]:.2f}W  '
      f'SAE={merged["sae"]:.4f}  F1={merged["f1"]:.3f}{f1s}')

diff_mae = merged['mae'] - RUN6_TEST_MAE
print(f'\n  vs Run6({RUN6_TEST_MAE:.2f}W): {"↓" if diff_mae < 0 else "↑"}{abs(diff_mae):.2f}W')

# ── 가전별 지표 ───────────────────────────────────────────────
print('\n가전별 Test 지표:')
for g in ['fast', 'slow', 'always_on']:
    g_names = [APPLIANCE_LABELS[i] for i in sorted(GROUP_INDICES[g])]
    cfg = SPEED_GROUP_CONFIG[g]
    print(f'\n  [{g} | {cfg["resample_hz"]}Hz@{cfg["window_size"]}]')
    for name in g_names:
        pa = merged['per_appliance'].get(name)
        if pa and pa.get('mae') is not None:
            f1_str = f"  f1={pa['f1']:.3f}" if pa.get('f1') is not None else ''
            print(f'    {name}: MAE={pa["mae"]:.1f}W{f1_str}')
        else:
            print(f'    {name}: null')

# ── 저장 ─────────────────────────────────────────────────────
merged.update({'split': 'test', 'run': 'run7'})
with open(RESULTS_DIR / 'run7_merged_test_metrics.json', 'w') as _f:
    _json.dump(merged, _f, ensure_ascii=False, indent=2)

final_summary = {
    'run6_ref':   {'val_mae': RUN6_VAL_MAE, 'test_mae': RUN6_TEST_MAE},
    'run7_merged': {'test_mae': merged['mae'], 'test_rmse': merged['rmse'],
                    'test_f1': merged['f1'], 'test_sae': merged['sae']},
    'run7_groups': {
        g: {
            'best_exp': summary[g]['best_exp'],
            'val_mae':  summary[g]['best_val_mae'],
        }
        for g in ['fast', 'slow', 'always_on']
    },
}
with open(RESULTS_DIR / 'run7_final_summary.json', 'w') as _f:
    _json.dump(final_summary, _f, ensure_ascii=False, indent=2)
print('\n저장: run7_merged_test_metrics.json / run7_final_summary.json')